# Live demo: heterogeneous parallel limit order book simulator

1. **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
2. The first cell installs MPI and builds the code with CUDA (takes a few minutes, so run it before the demo starts).
3. The second cell is the demo: tests, identical trades on every back end including the GPU, and the serial matching core.
4. The last cell plots simulated prices.

Colab has only 2 CPU threads, so CPU speedups here are small; the laptop shows CPU scaling.

In [ ]:
import os, subprocess

def sh(cmd):
    """Run a shell command, show its output, and stop if it fails."""
    print("$", cmd)
    r = subprocess.run("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError(f"command failed (exit {r.returncode}): {cmd}")

os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"          # Colab runs as root
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh("apt-get -qq update && apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null && mpirun --version | head -1")
os.chdir("/content")
sh("rm -rf /content/lob && git clone -q https://github.com/abhisaarguptaa/Stock-market-simulator-using-PCAP.git /content/lob")
os.chdir("/content/lob")
sh("cmake -S . -B build -DLOB_USE_CUDA=ON -DLOB_USE_MPI=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null && cmake --build build -j 2>&1 | tail -2")
print("Ready.")

## The demo

In [ ]:
!cd /content/lob && bash scripts/demo.sh build

## What the market looks like

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
os.chdir("/content/lob")
sh("./build/lobsim --agents 16000 --instruments 4 --ticks 20000 --decide cuda --match omp --prices /content/prices.csv > /dev/null")
p = pd.read_csv("/content/prices.csv")
fig, ax = plt.subplots(figsize=(10, 4))
for i, g in p.groupby("instrument"):
    ax.plot(g["tick"], g["price"], lw=0.8, label=f"instrument {i}")
ax.set_xlabel("tick"); ax.set_ylabel("price (ticks)"); ax.set_title("Simulated prices: 16,000 agents decided on the GPU, 20,000 ticks")
ax.legend(); plt.show()